In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os 
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import etl.load as load
import etl.extract as extract
import etl.fleet as fleet
import etl.schedule as schedule

In [23]:
TIPOS_DIAS = {
    "uteis": "307679",
    "sabado": "307681", 
    "domingo": "307680",
}

In [28]:
df['tipo dia'].unique()

array([ 8,  9, 12,  7,  1, 36, 10, 30, 31, 13, 24, 29, 14, 38])

In [29]:
linhas_move = load.linhas_move

df = load.load_mco_table(r'./base_dados/mco_move', filtro_linhas=linhas_move, sep=';', encoding='latin1')
df = df[df['tipo dia'] == 8]

In [6]:
dict_gtfs = load.load_gtfs_tables(r'./base_dados/gtfsbhtrans', ['routes', 'trips', 'calendar', 'stop_times'])

In [8]:
trips = extract.get_trips_for_routes(dict_gtfs['trips'], dict_gtfs['routes'], route_short_names=linhas_move)

Index(['viagem', 'linha', 'sublinha', 'pc', 'concessionaria', 'saida',
       'veiculo', 'chegada', 'catraca saida', 'catraca chegada', 'ocorrencia',
       'justificativa', 'tipo dia', 'extensao', 'falha mecanica',
       'evento inseguro', 'indicador fechamento', 'data fechamento',
       'total usuarios', 'empresa operadora', 'unnamed: 20'],
      dtype='str')

In [24]:
horarios = schedule.horarios_saida(trips, dict_gtfs['stop_times'],dict_service_id=TIPOS_DIAS)

Service_id não informado, considerando dia útil


NameError: name 'TIPOS_DIAS' is not defined

In [17]:
headway = schedule.headway_por_hora(horarios)

In [18]:
headway

,hora,route_short_name,n_partidas,headway_min
0,0,10,3,20.0
1,0,51,3,20.0
2,0,5106,1,60.0
3,0,5201,1,60.0
4,0,5250,3,20.0
...,...,...,...,...
524,23,8101,3,20.0
525,23,82,2,30.0
526,23,8251,1,60.0
527,23,83P,2,30.0


In [21]:
fleet.frota_por_demanda_todas_linhas(
    headway,
    df
)

,linha,frota_necessaria,hora_pico_frequencia,n_partidas_pico,tempo_ciclo_pico_min
0,10,2,00,3,39.299708
1,1031,9,07,7,76.867439
2,50,19,07,24,47.280982
3,51,25,07,20,72.051655
4,5106,11,12,9,72.179334
5,5107,15,06,14,60.993351
6,52,9,06,11,45.138072
7,5201,28,06,23,71.347939
8,5250,32,07,26,72.889696
9,5401,9,06,7,72.050742
